# Phase 12 — decision-setting study, reproducible analysis

Recomputes every published statistic from the released files alone. If you have the
release bundle (`responses.jsonl`, `design.json`, `scales.json`), you can run this without
the repository's private data.

**Read this before drawing conclusions.**

1. The design is **incomplete**: no rater saw the same case twice, so most (case, system)
   cells are rated once. Inter-rater agreement is estimable only on the anchor cells that
   every rater rated.
2. **Do not report a Likert mean without an agreement statistic beside it.**
3. Friedman's test needs complete blocks and this design has none. We block by rater for
   the Friedman, and use Durbin's test — Friedman's generalisation to a balanced incomplete
   block design — on the case blocks. Both are reported. Where they disagree, that is the
   finding.
4. Rows with `timing_source == "server"` were timed by a fallback clock that cannot
   subtract time spent with the tab hidden. **Their times are upper bounds.**


In [ ]:
from pathlib import Path
import json

import pandas as pd

RELEASE = Path("../artifacts/human_study/release")

rows = [json.loads(l) for l in (RELEASE / "responses.jsonl").read_text().splitlines() if l.strip()]
df = pd.DataFrame(rows)
print(f"{len(df)} responses, {df['rater'].nunique()} raters, {df['system'].nunique()} systems")
df.head()

## 0. Data quality — read this first

How much of the timing data is trustworthy, and how much of the design was realised.

In [ ]:
print("timing source:")
print(df["timing_source"].value_counts().to_string())

server = (df["timing_source"] == "server").mean()
if server:
    print(f"\n{server:.1%} of rows are upper-bound times. Consider restricting:")
    print("    df = df[df.timing_source == 'browser']")

print("\nitems per system:")
print(df.groupby("system").size().to_string())
print("\ncases per system (the gate's number):")
print(df.groupby("system")["case_id"].nunique().to_string())
print(f"\nrepeat items: {int(df['is_repeat'].sum())}")

## 1. Time-to-usable-draft

The headline. A measured reduction in drafting time against the template baseline is a
deployment claim; the rating scales are not.

Paired **by rater** — each rater contributes their mean under each system. Pairing by case
does not exist in this design, because no rater saw a case under two systems.

In [ ]:
from g2t_aml.human.study_analysis import normalised_levenshtein

BASELINE = "Bronze"
main = df[~df["is_repeat"]].copy()
main["edit_distance"] = [
    normalised_levenshtein(a, b)
    for a, b in zip(main["presented_narrative"], main["corrected_narrative"])
]

per_system = main.groupby("system")["seconds_to_usable_draft"].agg(["count", "mean", "median", "std"])
per_system.sort_values("mean")

In [ ]:
from scipy import stats

by_rater = main.pivot_table(
    index="rater", columns="system", values="seconds_to_usable_draft", aggfunc="mean"
)

for system in [s for s in by_rater.columns if s != BASELINE]:
    pair = by_rater[[system, BASELINE]].dropna()
    diff = pair[system] - pair[BASELINE]
    stat, p = stats.wilcoxon(pair[system], pair[BASELINE])
    pct = 100 * diff.mean() / pair[BASELINE].mean()
    print(f"{system:8s} {diff.mean():+8.1f}s ({pct:+5.1f}%)  n={len(pair):2d}  W={stat:6.1f}  p={p:.4f}")

## 2. Edit distance

How much of the draft survived contact with an expert. Normalised character-level Levenshtein.

In [ ]:
display(main.groupby("system")["edit_distance"].agg(["count", "mean", "median", "std"]).sort_values("mean"))

by_rater_edit = main.pivot_table(index="rater", columns="system", values="edit_distance", aggfunc="mean")
for system in [s for s in by_rater_edit.columns if s != BASELINE]:
    pair = by_rater_edit[[system, BASELINE]].dropna()
    stat, p = stats.wilcoxon(pair[system], pair[BASELINE])
    diff = (pair[system] - pair[BASELINE]).mean()
    print(f"{system:8s} {diff:+8.4f}  n={len(pair):2d}  W={stat:6.1f}  p={p:.4f}")

## 3. Would you file this after review?

A decision rather than an opinion.

In [ ]:
main.groupby("system")["would_file"].agg(["count", "mean"]).rename(
    columns={"mean": "filing_rate"}
).sort_values("filing_rate", ascending=False)

## 4. The rating scales — never without their agreement statistic

Agreement is computed over the **anchor cells**: the (case, system) cells that every rater
rated. In an incomplete design these are the only pairable units, and they are why the
design reserves them at all.

In [ ]:
from g2t_aml.human.study_analysis import krippendorff_alpha_ordinal

DIMENSIONS = ["factual_correctness", "completeness", "actionability", "readability", "regulatory_tone"]

cell_sizes = main.groupby(["case_id", "system"]).size()
anchors = cell_sizes[cell_sizes >= 2].index
print(f"{len(anchors)} doubly-rated cells available for agreement\n")

for dim in DIMENSIONS:
    units = [
        main[(main.case_id == c) & (main.system == s)][dim].astype(int).tolist()
        for c, s in anchors
    ]
    if units:
        alpha, lo, hi = krippendorff_alpha_ordinal(units)
        print(f"{dim:22s} alpha={alpha:.3f}  95% CI [{lo:.3f}, {hi:.3f}]  n_units={len(units)}")
    else:
        print(f"{dim:22s} NOT ESTIMABLE - no cell was rated twice")

In [ ]:
means = main.groupby("system")[DIMENSIONS].mean().T
means["_"] = ""
print("Report these ONLY alongside the alphas above.")
means

## 5. Intra-rater reliability

From the planted repeats. A check on the panel, not on the systems, and never reported per person.

In [ ]:
from g2t_aml.human.study_analysis import _alpha_ordinal

reps = df[df["is_repeat"]]
firsts = df[~df["is_repeat"]].drop_duplicates(["rater", "case_id"]).set_index(["rater", "case_id"])
pairs = [
    (firsts.loc[(r["rater"], r["case_id"])], r)
    for _, r in reps.iterrows()
    if (r["rater"], r["case_id"]) in firsts.index
]
print(f"{len(pairs)} repeat pairs\n")
for dim in DIMENSIONS:
    units = [[int(a[dim]), int(b[dim])] for a, b in pairs]
    mad = sum(abs(u[0] - u[1]) for u in units) / len(units)
    print(f"{dim:22s} alpha={_alpha_ordinal(units):.3f}  mean|diff|={mad:.2f}")

## 6. Omnibus tests and the critical-difference diagram

Friedman on rater-blocked means (the brief's test, underpowered at this panel size), and
Durbin on the case-blocked incomplete design (design-appropriate, uses every observation).

In [ ]:
from g2t_aml.human.study_analysis import (
    critical_difference_diagram, durbin_test, friedman_test, nemenyi_posthoc,
)

systems = sorted(main["system"].unique())

fried = friedman_test(
    {r: dict(row.dropna()) for r, row in by_rater.iterrows() if row.notna().all()}, systems
)
print(f"Friedman (rater-blocked): chi2={fried.statistic:.3f}  p={fried.p_value:.5f}  N={fried.n_blocks}")

case_blocks = {
    c: dict(g.groupby("system")["seconds_to_usable_draft"].mean())
    for c, g in main.groupby("case_id")
}
modal = pd.Series({c: len(v) for c, v in case_blocks.items()}).mode()[0]
balanced = {c: v for c, v in case_blocks.items() if len(v) == modal}
try:
    durb = durbin_test(balanced, systems)
    print(f"Durbin (case-blocked):    T={durb.statistic:.3f}  p={durb.p_value:.5f}  N={durb.n_blocks}")
except ValueError as exc:
    print(f"Durbin not run: {exc}")

In [ ]:
post = nemenyi_posthoc(fried.mean_ranks, fried.n_blocks)
print(f"CD = {post.critical_difference:.3f}")
print("significantly different pairs:", post.significant_pairs or "none")

critical_difference_diagram(post, Path("cd_time.png"), title="Time-to-usable-draft")
from IPython.display import Image
Image("cd_time.png")

## 7. Automatic metric vs human judgement

**The figure that licenses every automatic number in the rest of the paper.** If the
project's Layer-2 faithfulness score does not track expert judgement of factual
correctness, the large automatic evaluation is measuring something else.

Needs `automatic_scores.json` (item id to score), which is not part of the public release
because it is derived from the repository's evaluation pipeline rather than from the raters.

In [ ]:
from g2t_aml.human.study_analysis import pearson_with_ci, spearman_with_ci

scores_path = RELEASE.parent / "automatic_scores.json"
if scores_path.is_file():
    auto = json.loads(scores_path.read_text())
    paired = [(auto[i], float(f)) for i, f in zip(main["item_id"], main["factual_correctness"]) if i in auto]
    xs, ys = [a for a, _ in paired], [b for _, b in paired]
    rho, r = spearman_with_ci(xs, ys), pearson_with_ci(xs, ys)
    print(f"n = {len(paired)}")
    print(f"Spearman rho = {rho.statistic:.3f}  95% CI [{rho.ci_low:.3f}, {rho.ci_high:.3f}]  p={rho.p_value:.2e}")
    print(f"Pearson  r   = {r.statistic:.3f}  95% CI [{r.ci_low:.3f}, {r.ci_high:.3f}]  p={r.p_value:.2e}")

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(xs, [y + (hash(str(i)) % 100) / 500 - 0.1 for i, y in enumerate(ys)], s=8, alpha=0.3)
    ax.set_xlabel("automatic Layer-2 factual score")
    ax.set_ylabel("human factual correctness (jittered)")
    ax.set_title(f"Spearman rho = {rho.statistic:.3f}")
    fig.tight_layout()
else:
    print(f"No automatic scores at {scores_path}. This is a Phase 12 gate requirement.")

## 8. Rater effects

Is the between-system difference still there once each rater's own severity is accounted
for? Raters differ in how they use a scale, and an incomplete design does not balance that
automatically.

In [ ]:
import statsmodels.formula.api as smf

model = smf.mixedlm("factual_correctness ~ C(system)", main, groups=main["rater"]).fit()
print(model.summary())